In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
import joblib
from sklearn.preprocessing import StandardScaler

In [23]:
columns = (    
    ['engine_id', 'cycle']    
    + [f'op_setting_{i}' for i in range(1, 4)]    
    + [f'sensor_{i}' for i in range(1, 22)]    
)    


In [24]:
path = r"..\Dataset\Raw"

In [25]:
train_fd2 = pd.read_csv(path + r"\train_FD002.txt", sep=r'\s+', header=None, names=columns)    
test_fd2  = pd.read_csv(path + r"\test_FD002.txt",  sep=r'\s+', header=None, names=columns)    
rul_fd2   = pd.read_csv(path + r"\RUL_FD002.txt",   sep=r'\s+', header=None, names=['actual_RUL'])    
rul_fd2['engine_id'] = np.arange(1, len(rul_fd2) + 1)

# Cap the RUL Target 

In [26]:
RUL_CAP = 125
def compute_piecewise_rul(df, cap=125):
    """
    Computes piecewise linear RUL target capped at specified threshold.
    """
    df = df.copy()
    df['max_cycle'] = df.groupby('engine_id')['cycle'].transform('max')
    df['RUL_uncapped'] = df['max_cycle'] - df['cycle']
    df['RUL'] = df['RUL_uncapped'].clip(upper=cap)
    return df

train_fd2 = compute_piecewise_rul(train_fd2, cap=RUL_CAP)

In [27]:
print(f"Capped RUL (max={RUL_CAP}) calculated successfully. Value counts at cap:")
print((train_fd2['RUL'] == RUL_CAP).sum(), "cycles in healthy plateau.")

Capped RUL (max=125) calculated successfully. Value counts at cap:
21259 cycles in healthy plateau.


# Operating Condition Clustering (K-Means)

In [28]:
op_cols = ['op_setting_1', 'op_setting_2', 'op_setting_3']

# Fit KMeans on FD002 training operating settings
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
train_fd2['regime'] = kmeans.fit_predict(train_fd2[op_cols])

# Apply same clustering to FD002 test data
test_fd2['regime'] = kmeans.predict(test_fd2[op_cols])

# Save the KMeans model — this will be REUSED for FD004
os.makedirs(r'..\Dataset\Processed\FD002', exist_ok=True)
joblib.dump(kmeans, r'..\Dataset\Processed\FD002\cluster_model.pkl')

['..\\Dataset\\Processed\\FD002\\cluster_model.pkl']

In [29]:
print("FD002 Operating Condition Clustering complete:")
print("Train regime distribution:\n", train_fd2['regime'].value_counts().sort_index())
print("Test regime distribution:\n", test_fd2['regime'].value_counts().sort_index())

FD002 Operating Condition Clustering complete:
Train regime distribution:
 regime
0    13458
1     8122
2     8002
3     8044
4     8096
5     8037
Name: count, dtype: int64
Test regime distribution:
 regime
0    8483
1    5063
2    5042
3    5148
4    5107
5    5148
Name: count, dtype: int64


# Train/Validation Split (Engine-Level)
Perform engine-level 80/20 split before normalization and smoothing to strictly prevent data leakage.

In [30]:
# Engine-level split (80% train, 20% validation) to prevent data leakage
np.random.seed(42)
all_engines = train_fd2['engine_id'].unique()
train_engines = np.random.choice(all_engines, size=int(0.8 * len(all_engines)), replace=False)
val_engines = np.setdiff1d(all_engines, train_engines)

train_split_fd2 = train_fd2[train_fd2['engine_id'].isin(train_engines)].copy()
val_split_fd2   = train_fd2[train_fd2['engine_id'].isin(val_engines)].copy()

In [31]:
print(f"Total engines: {len(all_engines)}")
print(f"Train engines: {len(train_engines)} ({len(train_split_fd2)} cycles)")
print(f"Validation engines: {len(val_engines)} ({len(val_split_fd2)} cycles)")
assert len(np.intersect1d(train_engines, val_engines)) == 0, "Error: Engine overlap detected!"

Total engines: 260
Train engines: 208 (42889 cycles)
Validation engines: 52 (10870 cycles)


# Per-Regime Z-Score Normalization
StandardScaler is fitted strictly on the training engines (train_split_fd2) to prevent validation data leakage.

In [32]:
selected_sensors = [
    'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8',
    'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14',
    'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21'
]

In [33]:
regime_scalers = {}
# Fit separate scaler per regime strictly on training split to prevent data leakage
for regime in range(6):
    mask = train_split_fd2['regime'] == regime
    scaler = StandardScaler()
    train_split_fd2.loc[mask, selected_sensors] = scaler.fit_transform(
        train_split_fd2.loc[mask, selected_sensors]
    )
    regime_scalers[regime] = scaler

C:\Users\muiva\AppData\Local\Temp\ipykernel_16248\3960672909.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.76215989 -1.47079193 -0.76215989 ...  2.07236826  2.07236826
  1.36373622]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train_split_fd2.loc[mask, selected_sensors] = scaler.fit_transform(


In [34]:
# Apply corresponding scalers to FD002 validation and test data
for regime in range(6):
    mask_val = val_split_fd2['regime'] == regime
    if mask_val.any():
        val_split_fd2.loc[mask_val, selected_sensors] = regime_scalers[regime].transform(
            val_split_fd2.loc[mask_val, selected_sensors]
        )
        
    mask_test = test_fd2['regime'] == regime
    if mask_test.any():
        test_fd2.loc[mask_test, selected_sensors] = regime_scalers[regime].transform(
            test_fd2.loc[mask_test, selected_sensors]
        )

C:\Users\muiva\AppData\Local\Temp\ipykernel_16248\1715549838.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[-0.05352785  0.65510418 -0.76215989 ...  2.07236826  2.07236826
  2.07236826]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  val_split_fd2.loc[mask_val, selected_sensors] = regime_scalers[regime].transform(
C:\Users\muiva\AppData\Local\Temp\ipykernel_16248\1715549838.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 0.65510418 -0.76215989 -0.76215989 ...  0.65510418 -0.05352785
 -0.05352785]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  test_fd2.loc[mask_test, selected_sensors] = regime_scalers[regime].transform(


In [35]:
# Save scalers for deployment & evaluation
joblib.dump(regime_scalers, r'..\Dataset\Processed\FD002\regime_scalers.pkl')

# Verification check: mean should be ~0, std should be ~1 within each regime on training split
sample_check = train_split_fd2[train_split_fd2['regime'] == 0][selected_sensors].mean()
print("Verification - Mean of sensors in Regime 0 after normalization (should be ~0):")
print(sample_check.round(4).head(4))

Verification - Mean of sensors in Regime 0 after normalization (should be ~0):
sensor_2    0.0
sensor_3   -0.0
sensor_4   -0.0
sensor_7   -0.0
dtype: float64


# Noise Reduction (Rolling Smoothing)

In [36]:
def apply_sensor_smoothing(df, feature_cols, window=5):
    """
    Applies exponential or rolling mean smoothing per engine to reduce sensor noise.
    """
    df = df.copy()
    for col in feature_cols:
        df[col] = (
            df.groupby('engine_id')[col]
            .transform(lambda s: s.rolling(window=window, min_periods=1).mean())
        )
    return df

In [37]:
train_split_fd2 = apply_sensor_smoothing(train_split_fd2, selected_sensors, window=5)
val_split_fd2   = apply_sensor_smoothing(val_split_fd2,   selected_sensors, window=5)
test_fd2        = apply_sensor_smoothing(test_fd2,        selected_sensors, window=5)
print("Sensor rolling mean smoothing (window=5) applied successfully.")

Sensor rolling mean smoothing (window=5) applied successfully.


In [38]:
train_split_fd2.head()

,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,max_cycle,RUL_uncapped,RUL,regime
0,1,1,34.9983,0.8400,100.0,449.44,-1.094134,-1.449652,0.761195,5.48,...,0.02,-0.174348,2223,100.00,-0.883280,-1.341923,149,148,125,5
1,1,2,41.9982,0.8408,100.0,445.00,-0.323726,-0.836092,0.246594,3.91,...,0.02,-0.468254,2212,100.00,-1.259204,-1.328859,149,147,125,0
2,1,3,24.9988,0.6218,60.0,462.54,0.195368,-0.936634,0.020846,7.05,...,0.02,0.112839,1915,84.93,-1.377214,-0.311570,149,146,125,2
3,1,4,42.0077,0.8416,100.0,445.00,0.032661,-0.722333,-0.031736,3.91,...,0.02,-0.283069,2212,100.00,-1.024269,0.226538,149,145,125,0
4,1,5,25.0005,0.6203,60.0,462.54,0.138868,-0.769201,-0.097371,7.05,...,0.02,0.028550,1915,84.93,-1.053102,0.089689,149,144,125,2


# Create Sliding Window Sequences

In [39]:
def create_training_sequences(df, feature_cols, sequence_length=30):
    """
    Generate sliding-window sequences for training or validation across all engines.
    Returns: X array (N, seq_len, num_features), y array (N,)
    """
    X, y = [], []
    for engine_id in df['engine_id'].unique():
        engine_subset = df[df['engine_id'] == engine_id].sort_values('cycle')
        features = engine_subset[feature_cols].values
        targets = engine_subset['RUL'].values
        
        for end_idx in range(sequence_length - 1, len(engine_subset)):
            start_idx = end_idx - sequence_length + 1
            X.append(features[start_idx:end_idx + 1])
            y.append(targets[end_idx])
            
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

In [40]:
def create_test_sequences(df, rul_df, feature_cols, sequence_length=30):
    """
    Extract the final sliding-window sequence of length 30 for each test engine,
    and pair with ground-truth actual RUL.
    Applies left-padding (repeating the first cycle) for engines with fewer cycles
    than sequence_length, preventing silent dropping of short test engines.
    """
    X_test, y_test, valid_engine_ids = [], [], []
    for engine_id in rul_df['engine_id'].values:
        engine_subset = df[df['engine_id'] == engine_id].sort_values('cycle')
        data = engine_subset[feature_cols].values
        if len(data) >= sequence_length:
            seq = data[-sequence_length:]
        else:
            # Left-pad short engines by repeating the earliest available cycle
            pad_length = sequence_length - len(data)
            pad = np.repeat(data[:1], pad_length, axis=0)
            seq = np.vstack([pad, data])

        actual_rul = rul_df.loc[rul_df['engine_id'] == engine_id, 'actual_RUL'].values[0]
        X_test.append(seq)
        y_test.append(min(actual_rul, 125))  # Evaluate on capped RUL
        valid_engine_ids.append(engine_id)
            
    return (
        np.array(X_test, dtype=np.float32),
        np.array(y_test, dtype=np.float32),
        np.array(valid_engine_ids, dtype=np.int32)
    )

In [41]:
SEQUENCE_LENGTH = 30
X_train_fd2, y_train_fd2 = create_training_sequences(train_split_fd2, selected_sensors, SEQUENCE_LENGTH)
X_val_fd2, y_val_fd2     = create_training_sequences(val_split_fd2,   selected_sensors, SEQUENCE_LENGTH)
X_test_fd2, y_test_fd2, test_ids_fd2 = create_test_sequences(test_fd2, rul_fd2, selected_sensors, SEQUENCE_LENGTH)

# Save Processed Data

In [42]:
save_dir_fd2 = r'..\Dataset\Processed\FD002'
os.makedirs(save_dir_fd2, exist_ok=True)

np.save(os.path.join(save_dir_fd2, 'x_train.npy'), X_train_fd2)
np.save(os.path.join(save_dir_fd2, 'y_train.npy'), y_train_fd2)
np.save(os.path.join(save_dir_fd2, 'x_val.npy'),   X_val_fd2)
np.save(os.path.join(save_dir_fd2, 'y_val.npy'),   y_val_fd2)
np.save(os.path.join(save_dir_fd2, 'x_test.npy'),  X_test_fd2)
np.save(os.path.join(save_dir_fd2, 'y_test.npy'),  y_test_fd2)
np.save(os.path.join(save_dir_fd2, 'test_engine_ids.npy'), test_ids_fd2)

print("All FD002 preprocessed arrays saved to", save_dir_fd2)

All FD002 preprocessed arrays saved to ..\Dataset\Processed\FD002
